In [ ]:
import numpy as np
import pandas as pd
import sklearn

In [ ]:
Face = pd.read_csv("data/WM_Face_2bk_cleaned.csv")
Place = pd.read_csv("data/WM_Place_2bk_cleaned.csv")

In [ ]:
Face

In [ ]:
Face.iloc[:, 9:368]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X = Face.iloc[:, 9:368]
y = Face['BIS']

# Train/test split (e.g. 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LASSO regression with built-in cross-validation
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

# Model evaluation
print("Test R^2 score:", lasso.score(X_test_scaled, y_test))

# Inspect the regression coefficient of each brain region
# Get the brain-region names
brain_regions = X.columns  # X is Face.iloc[:, 9:368], so these columns are the brain regions

# Build a DataFrame holding coefficients and region names
coef_df = pd.DataFrame({'Brain Region': brain_regions, 'Lasso Coefficient': lasso.coef_})

# Sort by absolute coefficient to see the most important regions
coef_df = coef_df.reindex(coef_df['Lasso Coefficient'].abs().sort_values(ascending=False).index)

# Print the top 10 most important regions
print(coef_df.head(10))

# Plot the top 20 most important regions
plt.figure(figsize=(12, 6))
plt.barh(coef_df['Brain Region'][:20], coef_df['Lasso Coefficient'][:20], color='b')
plt.xlabel("Lasso coefficient")
plt.ylabel("Brain region")
plt.title("Top 20 regions selected by LASSO")
plt.gca().invert_yaxis()  # put the most important region on top
plt.show()

# Using high performer prediction

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

In [ ]:
# Data is stored in the Face DataFrame
X = Face.iloc[:, 9:368]  # 360 brain regions
y_continuous = Face['BIS']  # continuous variable BIS

# 75th percentile of BIS (threshold for high performers)
threshold = np.percentile(y_continuous, 75)

# Create the binary target variable high_performer
Face['high_performer'] = (y_continuous >= threshold).astype(int)

# Update y (0 = low performer, 1 = high performer)
y = Face['high_performer']

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LASSO logistic regression (L1 regularization)
logistic_lasso = LogisticRegressionCV(
    Cs=np.logspace(-2, 2, 30),  # C (inverse of regularization strength)
    cv=5,                        # 5-fold cross-validation
    penalty='elasticnet',                # elastic-net penalty
    solver='saga',          # solver supporting L1 penalties
    l1_ratios=[0.5],  # 50% L1，50% L2
    random_state=42
)
logistic_lasso.fit(X_train_scaled, y_train)

# Best C (inverse of regularization strength)
best_C = logistic_lasso.C_[0]
print(f"Best C: {best_C}")

# Test-set accuracy
test_acc = logistic_lasso.score(X_test_scaled, y_test)
print(f"Test accuracy: {test_acc:.4f}")

# Get the logistic-regression coefficients
coef = logistic_lasso.coef_[0]  # binary classification, so coef_ has one row

# Build a DataFrame holding coefficients and region names
brain_regions = X.columns
coef_df = pd.DataFrame({'Brain Region': brain_regions, 'Lasso Coefficient': coef})

# Sort by absolute coefficient
coef_df = coef_df.reindex(coef_df['Lasso Coefficient'].abs().sort_values(ascending=False).index)

# Print the top 10 most important regions
print(coef_df.head(10))

# Plot the top 20 most important regions
plt.figure(figsize=(12, 6))
plt.barh(coef_df['Brain Region'][:20], coef_df['Lasso Coefficient'][:20], color='b')
plt.xlabel("LASSO logistic-regression coefficient")
plt.ylabel("Brain region")
plt.title("Top 20 regions selected by LASSO logistic regression")
plt.gca().invert_yaxis()  # put the most important region on top
plt.show()


# SVM, continuous BIS

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Extract features and target
X = Face.iloc[:, 9:368]  # activation of 360 brain regions
y = Face['BIS']  # target variable BIS

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (important for SVMs)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the SVR model
svr = SVR(kernel='rbf')  # RBF kernel

# Hyperparameter search via GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],  # regularization parameter
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]  # kernel coefficient
}
grid_search = GridSearchCV(svr, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# Best model
best_svr = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

# Predict
y_pred = best_svr.predict(X_test_scaled)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")


k-means clustering

In [ ]:
Face_der = pd.read_csv("data/WM_Face_2bk_fdr_results.csv")
Place_der = pd.read_csv("data/WM_Place_2bk_fdr_results.csv")

In [ ]:
Face_der

In [ ]:
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt

In [ ]:
# Extract the relevant columns for clustering
X = Face_der[['First_Mean', 'Second_Mean']]

# Perform k-means clustering
kmeans = KMeans(n_clusters=6, random_state=42)
Face_der['Cluster'] = kmeans.fit_predict(X)

# Plot the clusters
plt.figure(figsize=(8, 6))
for cluster in range(6):
    cluster_data = Face_der[Face_der['Cluster'] == cluster]
    plt.scatter(cluster_data['First_Mean'], cluster_data['Second_Mean'], label=f'Cluster {cluster}')
    
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=200, c='black', marker='X', label='Centroids')
plt.xlabel('First_Mean')
plt.ylabel('Second_Mean')
plt.title('K-Means Clustering (k=6)')
# plt.legend()
plt.show()

In [ ]:
# Extract the relevant columns for clustering
X = Place_der[['First_Mean', 'Second_Mean']]

# Perform k-means clustering
kmeans = KMeans(n_clusters=6, random_state=42)
Place_der['Cluster'] = kmeans.fit_predict(X)

# Plot the clusters
plt.figure(figsize=(8, 6))
for cluster in range(6):
    cluster_data = Place_der[Place_der['Cluster'] == cluster]
    plt.scatter(cluster_data['First_Mean'], cluster_data['Second_Mean'], label=f'Cluster {cluster}')
    
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=200, c='black', marker='X', label='Centroids')
plt.xlabel('First_Mean')
plt.ylabel('Second_Mean')
plt.title('K-Means Clustering (k=6)')
# plt.legend()
plt.show()